# Agent 1 — File Reception playground

Demonstrates `FileReceptionAgent` end-to-end:
validation → SHA-256 → MIME detection → SSE events → audit record.

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [ ]:
import asyncio

from settings import Settings
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from classiflow.ingesta.agents import FileReceptionAgent
from classiflow.ingesta.mime import detect_mime
from classiflow.shared.audit.service import AuditService
from classiflow.shared.database.base import Base
from classiflow.shared.database.repositories.audit import SqlAuditRepository
from classiflow.shared.events.broadcaster import EventBroadcaster

print("imports OK")

## 2 — Database setup

Creates the SQLite engine and ensures all tables exist.
The `session_factory` is reused by every section below.

In [ ]:
engine = create_async_engine(Settings.database_url, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print("database ready — classiflow.db")

In [ ]:
from sqlalchemy.ext.asyncio import AsyncSession


def make_agent(session: AsyncSession) -> FileReceptionAgent:
    return FileReceptionAgent(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=EventBroadcaster(),
        mime_detector=detect_mime,
    )


print("make_agent ready")

## 4 — Run with a valid PDF

The cell uses top-level `await` — that works in Jupyter without `asyncio.run()`.

In [ ]:
MINIMAL_PDF = (
    b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog >>\nendobj\n"
    b"xref\n0 1\n0000000000 65535 f\ntrailer\n<< /Size 1 >>\nstartxref\n9\n%%EOF"
)

async with session_factory() as session:
    result = await make_agent(session).run(
        job_id="demo-001", filename="sample.pdf", file_bytes=MINIMAL_PDF
    )
    await session.commit()

print("=== File state ===")
print(f"  passed          : {result.passed}")
print(f"  sha256          : {result.sha256}")
print(f"  detected_mime   : {result.detected_mime}")
print(f"  file_size_bytes : {result.file_size_bytes}")
print(f"  rejection_reason: {result.rejection_reason}")

## 5 — Inspect the audit record

In [ ]:
async with session_factory() as session:
    records = await SqlAuditRepository(session).list_for_job("demo-001")

print("=== Audit records ===")
for r in records:
    print(f"  event       : {r.event}")
    print(f"  agent       : {r.agent}")
    print(f"  duration_ms : {r.duration_ms} ms")
    print(f"  detail      : {r.detail}")
    print()

## 6 — Observe SSE events in real time

The agent emits `STARTED` then `PASSED`/`FAILED` through the `EventBroadcaster`.
Here we subscribe before calling `run()` so we catch both events.

In [ ]:
broadcaster2 = EventBroadcaster()
events = []


async def collect() -> None:
    async for event in broadcaster2.subscribe("demo-002"):
        events.append(event)
        print(f"  SSE → agent={event.agent}  status={event.status}")


async with session_factory() as session:
    agent2 = FileReceptionAgent(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster2,
        mime_detector=detect_mime,
    )

    collect_task = asyncio.create_task(collect())
    await asyncio.sleep(0)

    await agent2.run(job_id="demo-002", filename="sample.pdf", file_bytes=MINIMAL_PDF)
    await broadcaster2.close("demo-002")
    await collect_task
    await session.commit()

print(f"\ncollected {len(events)} events")

## 7 — Rejection cases

Agent rejects: missing file, empty bytes, and anything over 50 MB.

In [ ]:
cases = [
    ("no file", None),
    ("empty file", b""),
    ("oversized", b"x" * (51 * 1024 * 1024)),
]

print("=== Rejection cases ===")
for label, data in cases:
    async with session_factory() as session:
        r = await make_agent(session).run(
            job_id=f"demo-{label}", filename="test.pdf", file_bytes=data
        )
        await session.commit()
    print(f"  {label:12} → passed={r.passed}  reason='{r.rejection_reason}'")

## 8 — Run with a real file from disk

Drop any PDF, DOCX, or image into:
```
src/classiflow/playground/samples/
```
then run the cell — it processes every file in that folder.

In [ ]:
from pathlib import Path

import IPython.display as ipyd
from IPython.display import HTML

samples_dir = next(
    p / "samples"
    for p in [Path.cwd(), Path.cwd() / "src" / "classiflow" / "playground"]
    if (p / "samples").is_dir()
)
files = sorted(samples_dir.iterdir())
if not files:
    msg = f"No files in {samples_dir}. Drop a PDF/DOCX/image there first."
    raise FileNotFoundError(msg)

for file_path in files:
    file_bytes = file_path.read_bytes()

    async with session_factory() as session:
        repo = SqlAuditRepository(session)
        result = await FileReceptionAgent(
            audit=AuditService(repo),
            broadcaster=EventBroadcaster(),
            mime_detector=detect_mime,
        ).run(job_id="demo-real", filename=file_path.name, file_bytes=file_bytes)
        audit_records = await repo.list_for_job("demo-real")
        duration = audit_records[0].duration_ms if audit_records else None
        await session.commit()

    status_color = "#2e7d32" if result.passed else "#c62828"
    status_icon = "✅ PASSED" if result.passed else "❌ FAILED"
    sha_display = result.sha256[:16] + "…" if result.sha256 else "—"
    size_kb = f"{len(file_bytes) / 1024:.1f} KB"
    duration_str = f"{duration} ms" if duration is not None else "—"

    rows = [
        ("File", file_path.name),
        ("Size", size_kb),
        ("MIME", result.detected_mime or "—"),
        ("SHA-256 (prefix)", sha_display),
        ("Duration", duration_str),
        ("Rejection reason", result.rejection_reason or "—"),
    ]

    rows_html = "".join(
        f'<tr><td style="color:#555;padding:4px 12px 4px 0;white-space:nowrap">'
        f"{k}</td>"
        f'<td style="font-family:monospace;padding:4px 0">{v}</td></tr>'
        for k, v in rows
    )

    ipyd.display(
        HTML(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;
                    margin:8px 0;font-family:sans-serif;max-width:540px">
          <div style="font-size:1.1em;font-weight:bold;color:{status_color};
                      margin-bottom:10px">{status_icon}</div>
          <table style="border-collapse:collapse;width:100%">{rows_html}</table>
        </div>
        """)
    )